# 11785 HW3P2: Automatic Speech Recognition

Welcome to HW3P2. In this homework, you will be using the same data from HW1 but will be incorporating sequence models. We recommend you get familaried with sequential data and the working of RNNs, LSTMs and GRUs to have a smooth learning in this part of the homework.

Disclaimer: This starter notebook will not be as elaborate as that of HW1P2 or HW2P2. You will need to do most of the implementation in this notebook because, it is expected after 2 HWs, you will be in a position to write a notebook from scratch. You are welcomed to reuse the code from the previous starter notebooks but may also need to make appropriate changes for this homework. <br>
We have also given you 3 log files for the Very Low Cutoff (Levenshtein Distance = 30) so that you can observe how loss decreases.

Common errors which you may face


*   Shape errors: Half of the errors from this homework will account to this category. Try printing the shapes between intermediate steps to debug
*   CUDA out of Memory: When your architecture has a lot of parameters, this can happen. Golden keys for this is, (1) Reducing batch_size (2) Call *torch.cuda.empty_cache* often, even inside your training loop, (3) Call *gc.collect* if it helps and (4) Restart run time if nothing works







# Prelimilaries

You will need to install packages for decoding and calculating the Levenshtein distance

In [ ]:
# !pip install python-Levenshtein
# !git clone --recursive https://github.com/parlance/ctcdecode.git
# !pip install wget
# %cd ctcdecode
# !pip install .
# %cd ..

# !pip install torchsummaryX # We also install a summary package to check our model's forward before training

     |████████████████████████████████| 50 kB 3.4 MB/s eta 0:00:011
  Created wheel for python-Levenshtein: filename=python_Levenshtein-0.12.2-cp38-cp38-linux_x86_64.whl size=172338 sha256=6b97072ffee3cef77ad04fc135acd6460f1cf83108487724a448972ad6ce8dd1
  Stored in directory: /home/ubuntu/.cache/pip/wheels/d7/0c/76/042b46eb0df65c3ccd0338f791210c55ab79d209bcc269e2c7
Successfully built python-Levenshtein
Cloning into 'ctcdecode'...
remote: Enumerating objects: 1102, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 1102 (delta 16), reused 28 (delta 13), pack-reused 1063
Receiving objects: 100% (1102/1102), 780.91 KiB | 11.83 MiB/s, done.
Resolving deltas: 100% (529/529), done.
Submodule 'third_party/ThreadPool' (https://github.com/progschj/ThreadPool.git) registered for path 'third_party/ThreadPool'
Submodule 'third_party/kenlm' (https://github.com/kpu/kenlm.git) registered for path 'third_party/kenlm'
Cloning into '/home/u

# Libraries

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torchsummaryX import summary
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
#import phonemes
from sklearn.metrics import accuracy_score
import gc
import zipfile
import pandas as pd
from tqdm import tqdm
import os
import datetime
import torch.optim as optim
import torchaudio
from torchaudio import transforms
# imports for decoding and distance calculation
import ctcdecode
import Levenshtein
from ctcdecode import CTCBeamDecoder
from slugify import slugify
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device: ", device)

Device:  cuda


# Kaggle (TODO)

You need to set up your Kaggle and download the data

In [ ]:
import json

TOKEN = {"username":"*********","key":"********"}

!pip install kaggle==1.5.12
! mkdir -p .kaggle
! mkdir -p ./ & mkdir -p ./.kaggle & mkdir -p ./.kaggle/

with open('./.kaggle/kaggle.json', 'w') as file:
    json.dump(TOKEN, file)

# ! pip install --upgrade --force-reinstall --no-deps kaggle
! ls "./.kaggle"
! chmod 600 ./.kaggle/kaggle.json
! cp ./.kaggle/kaggle.json ./.kaggle/

! kaggle config set -n path -v ./

kaggle.json
cp: './.kaggle/kaggle.json' and './.kaggle/kaggle.json' are the same file
- path is now set to: ./


In [ ]:
# !kaggle competitions download -c 11-785-s22-hw3p2

100%|█████████████████████████████████████▊| 1.84G/1.84G [00:22<00:00, 88.2MB/s]
100%|██████████████████████████████████████| 1.84G/1.84G [00:22<00:00, 88.9MB/s]


In [ ]:
# !unzip -q ./competitions/11-785-s22-hw3p2/11-785-s22-hw3p2.zip -d ./


# Dataset and dataloading (TODO)

In [ ]:
# PHONEME_MAP is the list that maps the phoneme to a single character.
# The dataset contains a list of phonemes but you need to map them to their corresponding characters to calculate the Levenshtein Distance
# You final submission should not have the phonemes but the mapped string
# No TODOs in this cell

PHONEME_MAP = [
    " ",
    ".", #SIL
    "a", #AA
    "A", #AE
    "h", #AH
    "o", #AO
    "w", #AW
    "y", #AY
    "b", #B
    "c", #CH
    "d", #D
    "D", #DH
    "e", #EH
    "r", #ER
    "E", #EY
    "f", #F
    "g", #G
    "H", #H
    "i", #IH
    "I", #IY
    "j", #JH
    "k", #K
    "l", #L
    "m", #M
    "n", #N
    "N", #NG
    "O", #OW
    "Y", #OY
    "p", #P
    "R", #R
    "s", #S
    "S", #SH
    "t", #T
    "T", #TH
    "u", #UH
    "U", #UW
    "v", #V
    "W", #W
    "?", #Y
    "z", #Z
    "Z" #ZH
]

PHONEMES = ["", 'SIL',   'AA',    'AE',    'AH',    'AO',    'AW',    'AY',
            'B',     'CH',    'D',     'DH',    'EH',    'ER',    'EY',
            'F',     'G',     'HH',    'IH',    'IY',    'JH',    'K',
            'L',     'M',     'N',     'NG',    'OW',    'OY',    'P',
            'R',     'S',     'SH',    'T',     'TH',    'UH',    'UW',
            'V',     'W',     'Y',     'Z',     'ZH']

# Run the cells below to load the train and validation data

In [ ]:
# This cell is where your actual TODOs start
# You will need to implement the Dataset class by your own. You may also implement it similar to HW1P2 (dont require context)
# The steps for implementation given below are how we have implemented it.
# However, you are welcomed to do it your own way if it is more comfortable or efficient.

class LibriSamples(torch.utils.data.Dataset):

    def __init__(self, data_path, partition= "train"): # You can use partition to specify train or dev

        self.X_dir = data_path + "/" + partition + "/mfcc/" # TODO: get mfcc directory path
        self.Y_dir = data_path + "/" + partition + "/transcript/" # TODO: get transcript path

        self.X_files = os.listdir(self.X_dir)  #TODO: list files in the mfcc directory
        self.Y_files = os.listdir(self.Y_dir)  # TODO: list files in the transcript directory

        # TODO: store PHONEMES from phonemes.py inside the class. phonemes.py will be downloaded from kaggle.
        # You may wish to store PHONEMES as a class attribute or a global variable as well.
        self.PHONEMES = PHONEMES

        assert(len(self.X_files) == len(self.Y_files))

        pass

    def __len__(self):
        return len(self.X_files)

    def __getitem__(self, ind):

        # X = [] # TODO: Load the mfcc npy file at the specified index ind in the directory
        # Y = [] # TODO: Load the corresponding transcripts

        X_path = self.X_dir + self.X_files[ind]

        Y_path = self.Y_dir + self.Y_files[ind]

        load_data = np.load(Y_path)[1:-1]

        label_load = [self.PHONEMES.index(yy) for yy in load_data]

        Y = np.array(label_load)

        X_data = np.load(X_path)
        X_data = (X_data - X_data.mean(axis=0))/X_data.std(axis=0)
        X = X_data
        # X = torch.as_tensor(X)


        # Remember, the transcripts are a sequence of phonemes. Eg. np.array(['<sos>', 'B', 'IH', 'K', 'SH', 'AA', '<eos>'])
        # You need to convert these into a sequence of Long tensors
        # Tip: You may need to use self.PHONEMES
        # Remember, PHONEMES or PHONEME_MAP do not have '<sos>' or '<eos>' but the transcripts have them.
        # You need to remove '<sos>' and '<eos>' from the trancripts.
        # Inefficient way is to use a for loop for this. Efficient way is to think that '<sos>' occurs at the start and '<eos>' occurs at the end.

        Yy = torch.cuda.LongTensor(Y)  #TODO: Convert sequence of  phonemes into sequence of Long tensors

        return X, Yy

    def collate_fn(batch):

        batch_x = [torch.tensor(x) for x,y in batch]
        batch_y = [torch.tensor(y) for x,y in batch]

        batch_x_pad = pad_sequence(batch_x, batch_first=True) #TODO: pad the sequence with pad_sequence (already imported)
        lengths_x = [len(x) for x in batch_x] #TODO: Get original lengths of the sequence before padding

        batch_y_pad = pad_sequence(batch_y, batch_first=True)# TODO: pad the sequence with pad_sequence (already imported)
        lengths_y = [len(y) for y in batch_y] # TODO: Get original lengths of the sequence before padding

        return batch_x_pad, batch_y_pad, torch.tensor(lengths_x), torch.tensor(lengths_y)


# You can either try to combine test data in the previous class or write a new Dataset class for test data
class LibriSamplesTest(torch.utils.data.Dataset):

    def __init__(self, data_path, test_order, partition='test'): # test_order is the csv similar to what you used in hw1

        test_order_list = list(pd.read_csv(test_order).file) # TODO: open test_order.csv as a list
        self.X = test_order_list # TODO: Load the npy files from test_order.csv and append into a list
        self.X_dir = data_path + "/" + partition + "/mfcc/"
        # You can load the files here or save the paths here and load inside __getitem__ like the previous class

    def __len__(self):
        return len(self.X)

    def __getitem__(self, ind):
        # TODOs: Need to return only X because this is the test dataset
        X_path = self.X_dir + self.X[ind]
        X_data = np.load(X_path)
        X_data = (X_data - X_data.mean(axis=0))/X_data.std(axis=0)
        x = X_data

        return x


    def collate_fn(batch):
        batch_x = [torch.tensor(x) for x in batch]
        batch_x_pad = pad_sequence(batch_x, batch_first=True) # TODO: pad the sequence with pad_sequence (already imported)
        lengths_x = [len(x) for x in batch_x] # TODO: Get original lengths of the sequence before padding

        return batch_x_pad, torch.tensor(lengths_x)

# The  cell below display the batch size and sizes of train and val dataset

In [ ]:
batch_size = 128

root = './hw3p2_student_data/hw3p2_student_data/' #TODO: Where your hw3p2_student_data folder is

train_data = LibriSamples(root, 'train')
val_data = LibriSamples(root, 'dev')
test_data = LibriSamplesTest(root, './hw3p2_student_data/hw3p2_student_data/test/test_order.csv')

train_loader = torch.utils.data.DataLoader(train_data, batch_size= batch_size, shuffle=True, collate_fn=LibriSamples.collate_fn)
val_loader = torch.utils.data.DataLoader(val_data, batch_size= batch_size, shuffle=True, collate_fn=LibriSamples.collate_fn)
test_loader = torch.utils.data.DataLoader(test_data, batch_size= batch_size, shuffle=False, collate_fn=LibriSamplesTest.collate_fn)

print("Batch size: ", batch_size)
print("Train dataset samples = {}, batches = {}".format(train_data.__len__(), len(train_loader)))
print("Val dataset samples = {}, batches = {}".format(val_data.__len__(), len(val_loader)))
print("Test dataset samples = {}, batches = {}".format(test_data.__len__(), len(test_loader)))

Batch size:  128
Train dataset samples = 28539, batches = 223
Val dataset samples = 2703, batches = 22
Test dataset samples = 2620, batches = 21


In [ ]:
# Optional
# Test code for checking shapes and return arguments of the train and val loaders
for data in val_loader:
    x, y, lx, ly = data # if you face an error saying "Cannot unpack", then you are not passing the collate_fn argument
    print(x.shape, y.shape, lx.shape, ly.shape)
    break

torch.Size([128, 3243, 13]) torch.Size([128, 348]) torch.Size([128]) torch.Size([128])


# Model Configuration (TODO)

In [ ]:
"""
The cell contains the model/architecture used to get the last levenshetein distance submitted on kaggle
Below is brief description of the model:
Embedding layer:
nn.Conv1d(13,256,kernel_size=9,stride=1,bias=False),
                       nn.BatchNorm1d(256),
                       nn.GELU(),
                       nn.MaxPool1d(kernel_size=2,stride=2),
                       nn.Dropout(p=0.45),

                       nn.Conv1d(256,256,kernel_size=4, stride=1),
                       nn.BatchNorm1d(256),
                       nn.GELU(),
                       nn.MaxPool1d(kernel_size=2,stride=2),
                       nn.Dropout(p=0.45)
lstm layer:
self.lstm = nn.LSTM(256, 256, 5, batch_first=True, dropout=0.45, bidirectional=True)# TODO: # Create a single layer, uni-directional LSTM with hidden_size = 256

Classification layer:

 nn.Linear(512, 1024),
                    nn.GELU(),
                    nn.Dropout(p=0.45),

                    nn.Linear(1024, 512),
                    nn.GELU(),
                    nn.Dropout(p=0.45),

                    nn.Linear(512, 41))

"""

'\nThe cell contains the model/architecture used to get the last levenshetein distance submitted on kaggle\nBelow is brief description of the model:\nEmbedding layer: \nnn.Conv1d(13,256,kernel_size=9,stride=1,bias=False),\n                       nn.BatchNorm1d(256),\n                       nn.GELU(),\n                       nn.MaxPool1d(kernel_size=2,stride=2),\n                       nn.Dropout(p=0.45),\n            \n                       nn.Conv1d(256,256,kernel_size=4, stride=1),\n                       nn.BatchNorm1d(256),\n                       nn.GELU(),\n                       nn.MaxPool1d(kernel_size=2,stride=2),\n                       nn.Dropout(p=0.45) \nlstm layer:\nself.lstm = nn.LSTM(256, 256, 5, batch_first=True, dropout=0.45, bidirectional=True)# TODO: # Create a single layer, uni-directional LSTM with hidden_size = 256\n\nClassification layer:\n\n nn.Linear(512, 1024),\n                    nn.GELU(),\n                    nn.Dropout(p=0.45),\n\n                    nn

In [ ]:
from torch.nn.modules import dropout
class Network(nn.Module):

    def __init__(self): # You can add any extra arguments as you wish

        super(Network, self).__init__()

        # Embedding layer converts the raw input into features which may (or may not) help the LSTM to learn better
        # For the very low cut-off you dont require an embedding layer. You can pass the input directly to the  LSTM
#         self.transforms = nn.Sequential(

#         torchaudio.transforms.FrequencyMasking(freq_mask_param=2),
#         torchaudio.transforms.TimeMasking(time_mask_param=32)
# )
        self.embedding = nn.Sequential(
                       nn.Conv1d(13,256,kernel_size=9,stride=1,bias=False),
                       nn.BatchNorm1d(256),
                       nn.GELU(),
                       nn.MaxPool1d(kernel_size=2,stride=2),
                       nn.Dropout(p=0.45),

                       nn.Conv1d(256,256,kernel_size=4, stride=1),
                       nn.BatchNorm1d(256),
                       nn.GELU(),
                       nn.MaxPool1d(kernel_size=2,stride=2),
                       nn.Dropout(p=0.45)

        )

        self.lstm = nn.LSTM(256, 256, 5, batch_first=True, dropout=0.45, bidirectional=True)# TODO: # Create a single layer, uni-directional LSTM with hidden_size = 256
        # Use nn.LSTM() Make sure that you give in the proper arguments as given in https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html

        self.classification = nn.Sequential(
                    nn.Linear(512, 1024),
                    nn.GELU(),
                    nn.Dropout(p=0.45),

                    nn.Linear(1024, 512),
                    nn.GELU(),
                    nn.Dropout(p=0.45),

                    nn.Linear(512, 41)
        )


    def forward(self, x, length_x): # TODO: You need to pass atleast 1 more parameter apart from self and x

        # x is returned from the dataloader. So it is assumed to be padded with the help of the collate_fn
        # if train:
        #     x = self.transforms(x)
        x = torch.permute(x, (0, 2, 1))
        x = self.embedding(x)
        length_x = length_x.clamp(max=x.shape[2])
        x = torch.permute(x, (0, 2, 1))
        packed_input = pack_padded_sequence(x, length_x, batch_first=True, enforce_sorted=False)# TODO: Pack the input with pack_padded_sequence. Look at the parameters it requires

        out1, (out2, out3) = self.lstm(packed_input) # TODO: Pass packed input to self.lstm
        # As you may see from the LSTM docs, LSTM returns 3 vectors. Which one do you need to pass to the next function?
        out, lengths  = pad_packed_sequence(out1, batch_first=True) # TODO: Need to 'unpack' the LSTM output using pad_packed_sequence

        out = self.classification(out)# TODO: Pass unpacked LSTM output to the classification layer
        out =  out.log_softmax(dim=2) # Optional: Do log softmax on the output. Which dimension?

        return out, lengths  # TODO: Need to return 2 variables

model = Network().to(device)
print(model)
summary(model, x.to(device), lx) # x and lx are from the previous cell

Network(
  (embedding): Sequential(
    (0): Conv1d(13, 256, kernel_size=(9,), stride=(1,), bias=False)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.45, inplace=False)
    (5): Conv1d(256, 256, kernel_size=(4,), stride=(1,))
    (6): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): GELU()
    (8): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Dropout(p=0.45, inplace=False)
  )
  (lstm): LSTM(256, 256, num_layers=5, batch_first=True, dropout=0.45, bidirectional=True)
  (classification): Sequential(
    (0): Linear(in_features=512, out_features=1024, bias=True)
    (1): GELU()
    (2): Dropout(p=0.45, inplace=False)
    (3): Linear(in_features=1024, out_features=512, bias=True)
    (4): GELU()
    (5): Dropout(p=0.45, inplace=False)
    (6): Linea

,Kernel Shape,Output Shape,Params,Mult-Adds
Layer,,,,
0_embedding.Conv1d_0,"[13, 256, 9]","[128, 256, 3235]",29952.0,96894720.0
1_embedding.BatchNorm1d_1,[256],"[128, 256, 3235]",512.0,256.0
2_embedding.GELU_2,-,"[128, 256, 3235]",NaN,NaN
3_embedding.MaxPool1d_3,-,"[128, 256, 1617]",NaN,NaN
4_embedding.Dropout_4,-,"[128, 256, 1617]",NaN,NaN
5_embedding.Conv1d_5,"[256, 256, 4]","[128, 256, 1614]",262400.0,423100416.0
6_embedding.BatchNorm1d_6,[256],"[128, 256, 1614]",512.0,256.0
7_embedding.GELU_7,-,"[128, 256, 1614]",NaN,NaN
8_embedding.MaxPool1d_8,-,"[128, 256, 807]",NaN,NaN


# Training Configuration (TODO)

# Run the cell below to call the CTCLoss, optimizer, scheduler and decoder

In [ ]:
criterion = nn.CTCLoss()  #TODO: What loss do you need for sequence to sequence models?
# Do you need to transpose or permute the model output to find out the loss? Read its documentation
lr = 2e-3
epochs = 25
optimizer =  optim.Adam(model.parameters(), lr=lr) # TODO: Adam works well with LSTM (use lr = 2e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=(len(train_loader) * epochs))
decoder = CTCBeamDecoder(
    PHONEME_MAP,
    model_path=None,
    alpha=0,
    beta=0,
    cutoff_top_n=40,
    cutoff_prob=1.0,
    beam_width=15,
    num_processes=4,
    blank_id=0,
    log_probs_input=True
)
#TODO: Intialize the CTC beam decoder
#Check out https://github.com/parlance/ctcdecode for the details on how to implement decoding
#Do you need to give log_probs_input = True or False?

# Run the cell below to compute the levenshtein distance

In [ ]:
# this function calculates the Levenshtein distance

def calculate_levenshtein(h, y, lh, ly, decoder, PHONEME_MAP):

    # h - ouput from the model. Probability distributions at each time step
    # y - target output sequence - sequence of Long tensors
    # lh, ly - Lengths of output and target
    # decoder - decoder object which was initialized in the previous cell
    # PHONEME_MAP - maps output to a character to find the Levenshtein distance

    # TODO: You may need to transpose or permute h based on how you passed it to the criterion
    # Print out the shapes often to debug
    beam_results, beam_scores, timesteps, out_lens = decoder.decode(h, seq_lens=lh)

    # TODO: call the decoder's decode method and get beam_results and out_len (Read the docs about the decode method's outputs)
    # Input to the decode method will be h and its lengths lh
    # You need to pass lh for the 'seq_lens' parameter. This is not explicitly mentioned in the git repo of ctcdecode.

    batch_size = len(h) #TODO
    dist = 0

    for i in range(batch_size): # Loop through each element in the batch

        h_sliced = beam_results[i][0][:out_lens[i][0]] # TODO: Get the output as a sequence of numbers from beam_results
        # Remember that h is padded to the max sequence length and lh contains lengths of individual sequences
        # Same goes for beam_results and out_lens
        # You do not require the padded portion of beam_results - you need to slice it with out_lens
        # If it is confusing, print out the shapes of all the variables and try to understand
        values = [k for idx, k in enumerate(PHONEME_MAP)]
        h_string = [values[k] for k in h_sliced]  # TODO: MAP the sequence of numbers to its corresponding characters with PHONEME_MAP and merge everything as a single string
        h_string = "".join(h_string)


        y_sliced = y[i,:ly[i].item()] #TODO: Do the same for y - slice off the padding with ly

        y_values= [PHONEME_MAP[ii] for ii in y_sliced]
        y_string = "".join(y_values) #TODO: MAP the sequence of numbers to its corresponding characters with PHONEME_MAP and merge everything as a single string
        dist += Levenshtein.distance(h_string, y_string)

    dist/=batch_size

    return dist

In [ ]:
# Optional but recommended

for i, data in enumerate(train_loader, 0):

    # Write a test code do perform a single forward pass and also compute the Levenshtein distance
    # Make sure that you are able to get this right before going on to the actual training
    # You may encounter a lot of shape errors
    # Printing out the shapes will help in debugging
    # Keep in mind that the Loss which you will use requires the input to be in a different format and the decoder expects it in a different format
    # Make sure to read the corresponding docs about it
    x, y, lx, ly = data
    x = x.to(device)
    y = y.to(device)
    lx = lx.cpu()
    ly = ly.cpu()
    output,len_out = model(x,lx)
    dist = calculate_levenshtein(output, y, len_out, ly, decoder, PHONEME_MAP)

    print(dist)

    pass

    break # one iteration is enough

158.4453125


# Run the cell below to call the eval function on the validation dataset

In [ ]:
def eval():
    total_dist = 0
    model.eval()

    for i, data in enumerate(val_loader):

    # Write a test code do perform a single forward pass and also compute the Levenshtein distance
    # Make sure that you are able to get this right before going on to the actual training
    # You may encounter a lot of shape errors
    # Printing out the shapes will help in debugging
    # Keep in mind that the Loss which you will use requires the input to be in a different format and the decoder expects it in a different format
    # Make sure to read the corresponding docs about it
        x, y, lx, ly = data
        x = x.to(device)
        y = y.to(device)
        lx = lx.cpu()
        ly = ly.cpu()
        output,len_out = model(x,lx)
        dist = calculate_levenshtein(output, y, len_out, ly, decoder, PHONEME_MAP)
        total_dist += float(dist)
    return total_dist / len(val_loader)




In [ ]:
# Uncomment and run the cell below to load the pretrained weight/saved model

In [ ]:
# checkpoint = torch.load("./model.pth")
# model.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
# scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

# Run the cell below to train the model

In [ ]:
torch.cuda.empty_cache() # Use this often

# TODO: Write the model evaluation function if you want to validate after every epoch

# You are free to write your own code for model evaluation or you can use the code from previous homeworks' starter notebooks
# However, you will have to make modifications because of the following.
# (1) The dataloader returns 4 items unlike 2 for hw2p2
# (2) The model forward returns 2 outputs
# (3) The loss may require transpose or permuting


# Note that when you give a higher beam width, decoding will take a longer time to get executed
# Therefore, it is recommended that you calculate only the val dataset's Levenshtein distance (train not recommended) with a small beam width
# When you are evaluating on your test set, you may have a higher beam width
#epochs = 30
scaler = torch.cuda.amp.GradScaler()
total_loss = 0
for epoch in range(epochs):
    model.train()
    batch_bar = tqdm(total=len(train_loader), dynamic_ncols=True, leave=False, position=0, desc='Train')
    total_loss = 0
    for i, data in enumerate(train_loader):
        optimizer.zero_grad()
        x, y, lx, ly = data
        x = x.float().to(device)
        y = y.to(device)
        lx = lx.cpu()
        ly = ly.cpu()
        with torch.cuda.amp.autocast():
            output,len_out = model(x,lx)
            loss = criterion(output.permute((1,0,2)),y,len_out,ly)
        total_loss += float(loss)
    # tqdm lets you add some details so you can monitor training as you train.
        batch_bar.set_postfix(lr="{:.04f}".format(float(optimizer.param_groups[0]['lr'])))

        scaler.scale(loss).backward() # This is a replacement for loss.backward()
        scaler.step(optimizer) # This is a replacement for optimizer.step()
        scaler.update()
        scheduler.step()
        batch_bar.update()
    batch_bar.close()
    print("Epoch {}/{}: Total Loss {:.04f}, Learning Rate {:.04f}".format(
            epoch + 1,
            epochs,
            float(loss),
            float(optimizer.param_groups[0]['lr'])))
    torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict':scheduler.state_dict()
            }, "./model.pth")

    print(total_loss)
    avg_val_dist = eval()
    print(avg_val_dist)




Epoch 1/25: Total Loss 0.1697, Learning Rate 0.0000
36.98099362850189
5.937642045454546


Epoch 2/25: Total Loss 0.1647, Learning Rate 0.0000
36.977700024843216
5.795501893939394


Epoch 3/25: Total Loss 0.1661, Learning Rate 0.0000
36.96213053166866
5.797088068181818


Epoch 4/25: Total Loss 0.1714, Learning Rate 0.0000
36.97907513380051
5.816903409090909


Epoch 5/25: Total Loss 0.1728, Learning Rate 0.0000
36.97440156340599
5.796022727272727


...

Epoch 21/25: Total Loss 0.1628, Learning Rate 0.0001
36.87259967625141
5.851609848484849


Epoch 22/25: Total Loss 0.1881, Learning Rate 0.0002
37.007865607738495
5.7415956439393945


Epoch 23/25: Total Loss 0.1581, Learning Rate 0.0002
36.940313175320625
5.790861742424243


Epoch 24/25: Total Loss 0.1721, Learning Rate 0.0002
36.91439738869667
5.758120265151515


Epoch 25/25: Total Loss 0.1711, Learning Rate 0.0002
37.03319300711155
5.804711174242424


In [ ]:
torch.cuda.empty_cache()

# TODO: Write the model training code
# You are free to write your own code for training or you can use the code from previous homeworks' starter notebooks
# However, you will have to make modifications because of the following.
# (1) The dataloader returns 4 items unlike 2 for hw2p2
# (2) The model forward returns 2 outputs
# (3) The loss may require transpose or permuting

# Tip: Implement mixed precision training

# Submit to kaggle (TODO)

# Run the cell below to validate the train model and submit to kaggle

In [ ]:
# TODO: Write your model evaluation code for the test dataset
# You can write your own code or use from the previous homewoks' stater notebooks
# You can't calculate loss here. Why?
pred=[]
model.eval()
decoder_test = CTCBeamDecoder(
    PHONEME_MAP,
    beam_width=105,
    num_processes=4,
    log_probs_input=True)

for i, data in enumerate(test_loader):
    x,  lx = data[0],data[1]
    x = x.to(device)
    #y = y.to(device)
    lx = lx.cpu()
    #ly = ly.cpu()
    output,len_out = model(x,lx)
    beam_results, beam_scores, timesteps, out_lens = decoder_test.decode(output, seq_lens=len_out)
    for j in range(output.shape[0]):
        h_sliced = beam_results[j, 0, 0:out_lens[j][0]]
        h_string = ''.join([PHONEME_MAP[ii] for ii in h_sliced])
        pred.append(h_string)


  #dist = calculate_levenshtein(output, y, len_out, ly, decoder, PHONEME_MAP)
  # total_dist += float(dist)
  # return total_dist / len(val_loader)


In [ ]:
# TODO: Generate the csv file
df = pd.DataFrame(pred, columns=['predictions'])
df.index.name = 'id'
df.to_csv('./submission.csv')
!kaggle competitions submit -c 11-785-s22-hw3p2 -f submission.csv -m 'submission_after_early'



100%|█████████████████████████████████████████| 212k/212k [00:00<00:00, 225kB/s]
Successfully submitted to Automatic Speech Recognition (ASR)

In [ ]:
del model
torch.cuda.empty_cache()
!nvidia-smi

Sun Apr  3 01:02:47 2022       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 460.32.03    Driver Version: 460.32.03    CUDA Version: 11.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla P100-PCIE...  Off  | 00000000:00:04.0 Off |                    0 |
| N/A   45C    P0    35W / 250W |   8043MiB / 16280MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------